# Finish the run: the four outstanding GPU tasks

The main study in `reproduce.ipynb` completed, and its output is analysed in
`results/ANALYSIS.md`. Four things could not be settled without a GPU and a
dataset, and this notebook does exactly those and nothing else.

| Task | Question it answers | ~T4 time |
|---|---|---|
| A. External diagnostics | Are the `_0`/`_1` labels being read backwards? That would explain every sub-chance AUC | 5 min |
| B. Cohort overlap | Is the "external" set already inside the training cohort? | 5 min |
| C. ONNX retry | Get a latency measurement, or a real error instead of a swallowed one | 15 min |
| D. Learning curves | Is the 4.2M `full` model simply undertrained at 10 epochs? | 60-90 min |

**Attach all three:**

- `tawsifurrahman/tuberculosis-tb-chest-xray-dataset`
- `kmader/pulmonary-chest-xray-abnormalities`
- the previous run's notebook output, `wenhaolu49/notebook7981e10859`

The third one carries the trained checkpoints and the image cache, so tasks A
and C need no retraining and the cache rebuild is skipped. Its `results/` is
deliberately ignored: the repository copy has already had the misaligned
`prune.csv` repaired (docs/BUGS.md item 2) and restoring the raw output would
put the corruption back.

Settings: **GPU T4 x1**, **Internet on**. Nothing here is multi-GPU.

Tasks A, B and C are cheap. Task D is the expensive one and is gated behind a
flag, so you can run the quick diagnostics first and come back.

In [ ]:
import glob, json, os, shutil, subprocess, sys, time

REPO_URL = "https://github.com/AIscend-Research/lightweight-tb-net"
REPO_DIR = "/kaggle/working/lightweight-tb-net"

RUN_A_EXTERNAL = True     # label diagnostics, cheap
RUN_B_OVERLAP  = True     # perceptual-hash overlap, cheap
RUN_C_ONNX     = True     # needs checkpoints (restored or retrained)
RUN_D_CURVES   = True     # the expensive one: retrains baselines
LONG_EPOCHS    = 30       # the undertrained-model test for task D
SEEDS          = [0, 1, 2, 3, 4]

ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
elif not ON_KAGGLE:
    REPO_DIR = os.getcwd()

os.chdir(REPO_DIR)
SRC = os.path.join(REPO_DIR, "src")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                        text=True).stdout.strip()
print("repo:  ", REPO_DIR)
print("commit:", COMMIT)


def run(*cmd, check=True):
    cmd = [sys.executable] + list(cmd)
    print("$", " ".join(str(c) for c in cmd), flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail.append(line)
        print(line, end="")
    p.wait()
    print(f"[{(time.time() - t0) / 60:.1f} min, exit {p.returncode}]")
    if check and p.returncode != 0:
        raise RuntimeError("".join(tail[-40:]))
    return p.returncode


def reset(*names):
    """Delete a stage CSV before re-running it, so rows are replaced and not
    appended to the restored copy."""
    for n in names:
        f = os.path.join(REPO_DIR, "results", n)
        if os.path.exists(f):
            os.remove(f)
            print("cleared results/" + n)

def nomask(paths):
    """Montgomery ships lung masks whose filenames match the CXRs."""
    return [p for p in paths if "mask" not in p.lower()]


In [ ]:
# Datasets and the previous run's output.
TB_DATASET    = "/kaggle/input/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset"
MC_SZ_DATASET = "/kaggle/input/datasets/kmader/pulmonary-chest-xray-abnormalities"
PRIOR_RUN     = "/kaggle/input/notebooks/wenhaolu49/notebook7981e10859"


def find_dir(root, must_contain):
    if not os.path.isdir(root):
        return None
    for dirpath, dirnames, _ in os.walk(root):
        if all(m in dirnames for m in must_contain):
            return dirpath
    return None


DATA_PATH = EXTERNAL_PATH = None
if ON_KAGGLE:
    DATA_PATH = (find_dir(TB_DATASET, ["Normal", "Tuberculosis"])
                 or find_dir("/kaggle/input", ["Normal", "Tuberculosis"]))
    hits = nomask(glob.glob(f"{MC_SZ_DATASET}/**/MCUCXR_*.png", recursive=True)
                  + glob.glob(f"{MC_SZ_DATASET}/**/CHNCXR_*.png", recursive=True)) \
        or nomask(glob.glob("/kaggle/input/**/MCUCXR_*.png", recursive=True)
                  + glob.glob("/kaggle/input/**/CHNCXR_*.png", recursive=True))
    if hits:
        EXTERNAL_PATH = os.path.commonpath([os.path.dirname(h) for h in hits])
        print(f"external: {len(hits)} radiographs (masks excluded)")
else:
    DATA_PATH = os.path.join(REPO_DIR, "data")

print("DATA_PATH     =", DATA_PATH)
print("EXTERNAL_PATH =", EXTERNAL_PATH)


def restore(src, dst_name):
    """Copy a directory out of the read-only input mount into the workspace."""
    if src and os.path.isdir(src):
        shutil.copytree(src, os.path.join(REPO_DIR, dst_name), dirs_exist_ok=True)
        n = len(os.listdir(os.path.join(REPO_DIR, dst_name)))
        print(f"restored {dst_name}/ from {src} ({n} entries)")
        return True
    return False


# Checkpoints and the image cache come from the previous run; restoring the
# cache saves the four-minute rebuild and restoring the checkpoints is what
# lets tasks A and C run without retraining anything.
#
# results/ deliberately does NOT come from there. The repository copy has
# already had the misaligned prune.csv repaired (see docs/BUGS.md item 2), and
# restoring the raw output would reintroduce the corruption.
ckpt_src = next((d for d in (f"{PRIOR_RUN}/artifacts/checkpoints",
                             f"{PRIOR_RUN}/lightweight-tb-net/checkpoints")
                 if os.path.isdir(d)),
                next(iter(glob.glob("/kaggle/input/**/checkpoints",
                                    recursive=True)), None))
restore(ckpt_src, "checkpoints")
restore(next((d for d in (f"{PRIOR_RUN}/lightweight-tb-net/cache",)
              if os.path.isdir(d)), None), "cache")

n_ckpt = len(glob.glob(os.path.join(REPO_DIR, "checkpoints", "*.pth")))
have_cache = os.path.exists(f"{REPO_DIR}/cache/faithful.npy")
print(f"checkpoints: {n_ckpt} | cache restored: {have_cache}")

In [ ]:
# Splits: regenerate from the dataset when it is attached (deterministic, and
# the generator refuses to emit a contaminated set), otherwise fall back to the
# copy the previous run produced.
if DATA_PATH and os.path.isdir(DATA_PATH):
    run(f"{SRC}/make_splits.py", "--data-path", DATA_PATH,
        "--out", f"{REPO_DIR}/data_splits", "--seed", "42")
else:
    print("TB dataset not attached; using the split shipped in the repo")
    run(f"{SRC}/make_splits.py", "--verify-only", "--out", f"{REPO_DIR}/data_splits")

if not have_cache:
    assert DATA_PATH, "no cache restored and no TB dataset attached"
    run(f"{SRC}/build_cache.py", "--data-path", DATA_PATH, "--variant", "both")
else:
    print("cache already present, skipping the rebuild")

## Task A - External label diagnostics

The compact model scored AUC 0.32 on Montgomery and Shenzhen, well below the
0.5 a coin flip would give. A systematically reversed ranking is more often a
label or preprocessing defect than a domain-shift result.

This prints the parsed class balance against the published one (Montgomery 80
normal / 58 TB, Shenzhen 326 / 336) and writes a verdict to
`results/external_labels.csv`. **Read that verdict before anything else in
this notebook.** If it says the labels look inverted, the external result is a
bug and not a finding.

In [ ]:
if RUN_A_EXTERNAL and EXTERNAL_PATH and n_ckpt:
    reset("external.csv", "external_labels.csv")
    run(f"{SRC}/experiments.py", "--stage", "external", "--force",
        "--external-path", EXTERNAL_PATH, "--arch", "compact", "full",
        "--caches", "faithful", "--seeds", *map(str, SEEDS))

    import pandas as pd
    p = f"{REPO_DIR}/results/external_labels.csv"
    if os.path.exists(p):
        display(pd.read_csv(p))
elif RUN_A_EXTERNAL:
    print("skipped: need both the external dataset and checkpoints")

In [ ]:
# Eyeball the preprocessing: if external images come out looking unlike the
# training images, the auto-crop is the suspect rather than the labels.
if RUN_A_EXTERNAL and EXTERNAL_PATH:
    import matplotlib.pyplot as plt
    import numpy as np
    sys.path.insert(0, SRC)
    from build_cache import process_faithful

    ext = sorted(nomask(glob.glob(f"{EXTERNAL_PATH}/**/*_[01].png",
                                  recursive=True)))[:4]
    train_arr = np.load(f"{REPO_DIR}/cache/faithful.npy", mmap_mode="r")

    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for i, p in enumerate(ext):
        axes[0, i].imshow(process_faithful(p), cmap="gray", vmin=0, vmax=255)
        axes[0, i].set_title(os.path.basename(p), fontsize=7)
    for i in range(4):
        axes[1, i].imshow(train_arr[i * 211], cmap="gray", vmin=0, vmax=255)
        axes[1, i].set_title("training cohort", fontsize=7)
    for ax in axes.flat:
        ax.axis("off")
    fig.suptitle("External (top) against training (bottom), same pipeline")
    os.makedirs(f"{REPO_DIR}/figures", exist_ok=True)
    fig.savefig(f"{REPO_DIR}/figures/external_preprocessing_check.png",
                dpi=150, bbox_inches="tight")
    plt.show()

## Task B - Cohort overlap

The Rahman database aggregates several sources and the NLM Montgomery and
Shenzhen sets are among them. If those images are already in the training
split, the external evaluation is not external. Perceptual hashing (dHash,
Hamming distance <= 5) against every cached training image.

In [ ]:
if RUN_B_OVERLAP and EXTERNAL_PATH:
    reset("overlap.csv")
    run(f"{SRC}/experiments.py", "--stage", "overlap", "--force",
        "--external-path", EXTERNAL_PATH, "--caches", "faithful")

    import pandas as pd
    p = f"{REPO_DIR}/results/overlap.csv"
    if os.path.exists(p):
        df = pd.read_csv(p)
        real = df[df["external"] != "(none)"]
        print(f"{len(real)} external images match a training-cohort image")
        if len(real):
            display(real.groupby(["cohort", "matched_split"]).size()
                    .rename("matches").reset_index())
            display(real.head(10))

## Task C - ONNX export and latency

The previous run produced no `.onnx` files at all and therefore no
`latency.csv`, so every deployment-latency claim is currently unmeasured. The
original call used `opset_version=13` with `dynamic_axes=None` and the real
error was discarded because only `str(exc)` was printed.

The export now tries three configurations in order (legacy at opset 17, the
torch 2.6+ dynamo exporter, legacy at opset 13), prints the full traceback,
and records failures to `results/latency_failures.csv`. Either this produces
latency numbers or it produces a diagnosable error.

Run on CPU: half-precision convolution is not supported on CPU, so the FP16
rows are skipped there, and a shared cloud vCPU makes latency indicative
rather than device-representative.

In [ ]:
if RUN_C_ONNX:
    if not n_ckpt:
        print("no checkpoints: retraining baseline and prune first")
        run(f"{SRC}/experiments.py", "--stage", "baseline", "--force",
            "--arch", "compact", "--caches", "faithful",
            "--seeds", *map(str, SEEDS))
        run(f"{SRC}/experiments.py", "--stage", "prune", "--force",
            "--arch", "compact", "--caches", "faithful",
            "--seeds", *map(str, SEEDS))

    reset("quantize.csv", "latency.csv", "latency_failures.csv")
    run(f"{SRC}/experiments.py", "--stage", "quantize", "--force",
        "--arch", "compact", "full", "--caches", "faithful",
        "--seeds", *map(str, SEEDS))

    import pandas as pd
    for f in ("latency.csv", "latency_failures.csv"):
        p = f"{REPO_DIR}/results/{f}"
        print(f"\n--- {f} ---")
        display(pd.read_csv(p)) if os.path.exists(p) else print("absent")

## Task D - Learning curves and the undertrained hypothesis

The 4.2M-parameter `full` model scores below the 0.27M `compact` one (92.14
against 98.81 accuracy). Before that gets written up as a capacity result, it
has to be ruled out as a training-budget artifact: ten epochs was chosen
against the old contaminated splits and was never tuned for the larger model.

Two things happen here. Every run now logs per-epoch validation metrics to
`results/history.csv`, so you can see whether sensitivity was still climbing
at the last epoch. And `full` is trained again for `LONG_EPOCHS`, which tests
the hypothesis directly.

`epochs` is a column in `baseline.csv`, so the long arm is distinguishable
from the 10-epoch one and both survive in the same file.

In [ ]:
if RUN_D_CURVES:
    reset("baseline.csv", "history.csv")

    # 10 epochs, as before, but now recording learning curves.
    run(f"{SRC}/experiments.py", "--stage", "baseline", "--force",
        "--arch", "compact", "full", "--caches", "faithful", "simple",
        "--seeds", *map(str, SEEDS), "--epochs", "10")

    # The long arm: does the larger model catch up given more budget?
    run(f"{SRC}/experiments.py", "--stage", "baseline", "--force",
        "--arch", "full", "--caches", "faithful",
        "--seeds", *map(str, SEEDS), "--epochs", str(LONG_EPOCHS))

In [ ]:
# Learning curves: a line still rising at the right edge means undertrained.
p = f"{REPO_DIR}/results/history.csv"
if os.path.exists(p):
    import matplotlib.pyplot as plt
    import pandas as pd
    h = pd.read_csv(p)
    h["arch"] = h["tag"].str.split("/").str[0]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
    for arch, colour in (("compact", "#1f77b4"), ("full", "#d62728")):
        sub = h[h["arch"] == arch]
        if sub.empty:
            continue
        for ax, col in zip(axes, ("val_sens", "train_loss")):
            m = sub.groupby("epoch")[col].agg(["mean", "std"])
            ax.errorbar(m.index, m["mean"], yerr=m["std"].fillna(0),
                        fmt="-o", capsize=3, color=colour, label=arch)
    axes[0].set_ylabel("validation sensitivity (%)")
    axes[1].set_ylabel("training loss")
    for ax in axes:
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.suptitle("Learning curves: is the larger model still improving?")
    fig.savefig(f"{REPO_DIR}/figures/learning_curves.png", dpi=150,
                bbox_inches="tight")
    plt.show()

    last = h[h["epoch"] == h["epoch"].max()]
    print("\nfinal-epoch validation sensitivity by architecture:")
    print(last.groupby("arch")["val_sens"].agg(["mean", "std", "count"]))

## Regenerate the analysis and figures, then package

In [ ]:
json.dump({"commit": COMMIT, "repo": REPO_URL, "seeds": SEEDS,
           "archs": ["compact", "full"], "caches": ["faithful", "simple"],
           "data_path": DATA_PATH, "external_path": EXTERNAL_PATH,
           "tasks": {"external": RUN_A_EXTERNAL, "overlap": RUN_B_OVERLAP,
                     "onnx": RUN_C_ONNX, "curves": RUN_D_CURVES},
           "long_epochs": LONG_EPOCHS},
          open(f"{REPO_DIR}/results/run_manifest.json", "w"), indent=2)

run(f"{SRC}/experiments.py", "--stage", "summary", check=False)
run(f"{SRC}/analyze.py")
run(f"{SRC}/make_figures.py", check=False)

from IPython.display import Markdown, display
display(Markdown(open(f"{REPO_DIR}/results/ANALYSIS.md").read()))

In [ ]:
OUT = "/kaggle/working/artifacts" if ON_KAGGLE else f"{REPO_DIR}/artifacts"
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT, exist_ok=True)
for sub in ("results", "figures", "data_splits", "deploy_repro", "checkpoints"):
    src = os.path.join(REPO_DIR, sub)
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(OUT, sub), dirs_exist_ok=True)
shutil.make_archive(OUT, "zip", OUT)
print("packaged ->", OUT + ".zip")

print("\nfiles to bring back into the repo (small, and the whole point):")
for f in sorted(glob.glob(f"{OUT}/results/*.csv")
                + glob.glob(f"{OUT}/results/*.md")
                + glob.glob(f"{OUT}/figures/*.png")):
    print(f"  {os.path.relpath(f, OUT)}  ({os.path.getsize(f) / 1024:.0f} kB)")

## What to do with the output

Bring back `results/` and `figures/` only. Checkpoints stay in the zip and
belong on Zenodo, not in git.

Then read, in this order:

1. `results/external_labels.csv` - if the verdict is `LABELS LOOK INVERTED`,
   the sub-chance external AUC is a bug, and section 6 of the analysis has to
   be rewritten rather than published.
2. `results/overlap.csv` - any match at all means the external evaluation is
   partly in-distribution and its numbers are an upper bound.
3. `results/latency.csv`, or `latency_failures.csv` if the export failed
   again. The failure file now carries a real error message.
4. `figures/learning_curves.png` - if `full` is still climbing at epoch 10 but
   flattens by 30, its underperformance is a training-budget artifact and no
   capacity claim should be made from the 10-epoch numbers.